## Introduzione

L'Internet of Things (IoT) permette di raccogliere dati da dispositivi e sensori connessi alla rete. In questo progetto utilizziamo Apache Spark per generare, elaborare e analizzare dati provenienti da sensori IoT. Inoltre, viene simulato un flusso di dati in tempo reale utilizzando Spark Structured Streaming.

## Obiettivi

- Installare e configurare PySpark su Google Colab.
- Creare una sessione Spark.
- Generare dati simulati provenienti da sensori IoT.
- Analizzare i dati utilizzando Spark SQL.
- Calcolare statistiche come temperatura e umidità medie.
- Simulare l'elaborazione dei dati in tempo reale con Spark Streaming.

##  Installazione di PySpark

In questa fase installiamo la libreria PySpark, necessaria per utilizzare Apache Spark in ambiente Python all'interno di Google Colab.

In [25]:
!pip install pyspark

## Creazione della Spark Session

La Spark Session rappresenta il punto di accesso principale a tutte le funzionalità di Apache Spark. Senza di essa non è possibile creare DataFrame o eseguire operazioni sui dati.

In [26]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("IoT Project") \
    .getOrCreate()


Version Spark : 4.0.2


##  Verifica dell'installazione

Visualizziamo la versione di Spark per verificare che l'installazione sia avvenuta correttamente.

In [28]:
print("Version Spark :", spark.version)

Version Spark : 4.0.2


##  Creazione di un piccolo dataset di esempio

Viene creato un DataFrame contenente alcuni dati di sensori IoT. Questo passaggio serve per verificare il corretto funzionamento dell'ambiente Spark.

In [27]:
data = [
    ("S1", 25.5, 60),
    ("S2", 30.1, 45),
    ("S3", 22.0, 70)
]

df = spark.createDataFrame(
    data,
    ["sensor_id", "temperature", "humidity"]
)

df.show()

+---------+-----------+--------+
|sensor_id|temperature|humidity|
+---------+-----------+--------+
|       S1|       25.5|      60|
|       S2|       30.1|      45|
|       S3|       22.0|      70|
+---------+-----------+--------+



##  Generazione dei dati IoT

In questa fase vengono generati 10.000 dati simulati provenienti da sensori IoT. Ogni record contiene timestamp, ID del sensore, temperatura e umidità. I dati vengono poi salvati nel file `sensors.csv`.

In [29]:
import pandas as pd
import random
from datetime import datetime, timedelta

data = []
start = datetime(2026, 6, 1, 8, 0, 0)

for i in range(10000):
    timestamp = start + timedelta(seconds=i)
    sensor_id = "S" + str(random.randint(1, 10))
    temperature = round(random.uniform(15, 40), 2)
    humidity = round(random.uniform(20, 90), 2)

    data.append([timestamp, sensor_id, temperature, humidity])

df_pandas = pd.DataFrame(
    data,
    columns=["timestamp", "sensor_id", "temperature", "humidity"]
)

df_pandas.to_csv("sensors.csv", index=False)

df_pandas.head()

,timestamp,sensor_id,temperature,humidity
0,2026-06-01 08:00:00,S6,28.71,71.91
1,2026-06-01 08:00:01,S10,15.43,80.43
2,2026-06-01 08:00:02,S4,25.28,54.69
3,2026-06-01 08:00:03,S8,31.11,40.72
4,2026-06-01 08:00:04,S10,16.88,75.70


## Verifica della dimensione del dataset

Controlliamo quante righe e colonne contiene il dataset generato.

In [30]:
df_pandas.shape

(10000, 4)

## Caricamento e verifica del dataset

Apache Spark legge il file CSV e crea un DataFrame distribuito. Vengono visualizzati lo schema e alcune righe del dataset.

In [31]:
df = spark.read.csv(
    "sensors.csv",
    header=True,
    inferSchema=True
)

df.show(10)

+-------------------+---------+-----------+--------+
|          timestamp|sensor_id|temperature|humidity|
+-------------------+---------+-----------+--------+
|2026-06-01 08:00:00|       S6|      28.71|   71.91|
|2026-06-01 08:00:01|      S10|      15.43|   80.43|
|2026-06-01 08:00:02|       S4|      25.28|   54.69|
|2026-06-01 08:00:03|       S8|      31.11|   40.72|
|2026-06-01 08:00:04|      S10|      16.88|    75.7|
|2026-06-01 08:00:05|       S5|      29.38|   21.19|
|2026-06-01 08:00:06|       S9|      35.67|   26.43|
|2026-06-01 08:00:07|       S5|      20.22|   55.79|
|2026-06-01 08:00:08|       S2|      18.76|   31.08|
|2026-06-01 08:00:09|       S1|      20.98|   44.56|
+-------------------+---------+-----------+--------+
only showing top 10 rows


## Struttura del dataset

Visualizziamo il tipo di ogni colonna del DataFrame.

In [32]:
df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- sensor_id: string (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)



## Numero totale di record

Contiamo il numero totale di righe presenti nel dataset.

In [33]:
df.count()

10000

##  Creazione della vista SQL

Trasformiamo il DataFrame in una tabella temporanea per poter utilizzare query SQL.

In [34]:
df.createOrReplaceTempView("sensors")

##  Temperatura media per sensore

Calcoliamo la temperatura media registrata da ogni sensore.

In [12]:
spark.sql("""
SELECT sensor_id,
       ROUND(AVG(temperature),2) AS avg_temperature
FROM sensors
GROUP BY sensor_id
ORDER BY sensor_id
""").show()

+---------+---------------+
|sensor_id|avg_temperature|
+---------+---------------+
|       S1|          27.37|
|      S10|           27.8|
|       S2|          27.45|
|       S3|          27.59|
|       S4|          27.27|
|       S5|          27.37|
|       S6|          27.63|
|       S7|          27.25|
|       S8|          27.51|
|       S9|           27.9|
+---------+---------------+



##  Umidità media per sensore

Calcoliamo l'umidità media registrata da ogni sensore.

In [ ]:
spark.sql("""
SELECT sensor_id,
       ROUND(AVG(humidity),2) AS avg_humidity
FROM sensors
GROUP BY sensor_id
ORDER BY sensor_id
""").show()

+---------+------------+
|sensor_id|avg_humidity|
+---------+------------+
|       S1|       54.18|
|      S10|       55.97|
|       S2|       55.03|
|       S3|        55.8|
|       S4|       54.71|
|       S5|       55.54|
|       S6|       54.57|
|       S7|       55.66|
|       S8|       55.24|
|       S9|       54.04|
+---------+------------+



##  Rilevamento delle anomalie

Individuiamo tutte le misurazioni con temperatura superiore a 35°C.

In [14]:
spark.sql("""
SELECT *
FROM sensors
WHERE temperature > 35
ORDER BY temperature DESC
""").show(20)

+-------------------+---------+-----------+--------+
|          timestamp|sensor_id|temperature|humidity|
+-------------------+---------+-----------+--------+
|2026-06-01 08:08:08|       S3|       40.0|   60.68|
|2026-06-01 09:33:08|      S10|       40.0|   28.55|
|2026-06-01 10:04:53|       S9|       40.0|   41.24|
|2026-06-01 08:40:12|       S9|      39.99|   38.13|
|2026-06-01 09:00:44|      S10|      39.99|   80.87|
|2026-06-01 09:11:07|      S10|      39.99|   49.29|
|2026-06-01 09:22:57|       S3|      39.99|   35.24|
|2026-06-01 10:43:06|       S7|      39.99|   41.15|
|2026-06-01 10:15:38|       S1|      39.99|   24.77|
|2026-06-01 10:29:08|       S4|      39.99|   48.26|
|2026-06-01 10:38:27|       S2|      39.99|    30.7|
|2026-06-01 08:02:09|       S2|      39.98|   21.76|
|2026-06-01 08:41:55|       S7|      39.98|   51.01|
|2026-06-01 09:45:47|       S6|      39.98|   56.55|
|2026-06-01 10:36:21|       S4|      39.98|   65.48|
|2026-06-01 08:42:14|       S5|      39.97|   

##  Preparazione dello streaming

Creiamo una cartella per simulare l'arrivo dei dati in tempo reale.

In [15]:
import os
import shutil

shutil.rmtree("stream_data", ignore_errors=True)
os.makedirs("stream_data", exist_ok=True)

##  Definizione dello schema

Definiamo le colonne dei dati che arriveranno nello streaming.

In [16]:
from pyspark.sql.types import StructType, StringType, DoubleType

schema = StructType() \
    .add("timestamp", StringType()) \
    .add("sensor_id", StringType()) \
    .add("temperature", DoubleType()) \
    .add("humidity", DoubleType())

##  Lettura del flusso dati

Spark legge automaticamente i nuovi file CSV nella cartella.

In [17]:
stream_df = spark.readStream \
    .schema(schema) \
    .option("header", True) \
    .csv("stream_data")

##  Calcolo delle medie

Calcoliamo temperatura e umidità media per ogni sensore.

In [18]:
stream_result = stream_df.groupBy("sensor_id") \
    .avg("temperature", "humidity")

##  Avvio dello streaming

Avviamo lo streaming e salviamo i risultati in memoria.

In [19]:
query = stream_result.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("iot_stream_results") \
    .start()

## Primo batch

Aggiungiamo i primi 20 dati nella cartella streaming.

In [20]:
df_pandas.iloc[0:20].to_csv("stream_data/batch_1.csv", index=False)

## Risultati dopo il primo batch

Aspettiamo alcuni secondi e visualizziamo i risultati elaborati da Spark.

In [21]:
import time

time.sleep(5)

spark.sql("""
SELECT *
FROM iot_stream_results
ORDER BY sensor_id
""").show()

+---------+----------------+-------------+
|sensor_id|avg(temperature)|avg(humidity)|
+---------+----------------+-------------+
+---------+----------------+-------------+



## Secondo batch

Aggiungiamo altri 20 dati per aggiornare i risultati dello streaming.

In [ ]:
df_pandas.iloc[20:40].to_csv("stream_data/batch_2.csv", index=False)

time.sleep(5)

spark.sql("""
SELECT *
FROM iot_stream_results
ORDER BY sensor_id
""").show()

+---------+------------------+------------------+
|sensor_id|  avg(temperature)|     avg(humidity)|
+---------+------------------+------------------+
|       S1|29.709999999999997| 61.06666666666667|
|      S10|23.907500000000002|43.504999999999995|
|       S2|             16.02|             49.65|
|       S3|             37.01|             27.14|
|       S4|             21.29|             32.41|
|       S5| 29.25333333333333| 49.03333333333333|
|       S6|              17.1|             56.47|
|       S7|26.755000000000003|            48.695|
|       S8|27.560000000000002|53.709999999999994|
|       S9|             21.05|             40.24|
+---------+------------------+------------------+



## Arresto dello streaming

Fermiamo il processo di streaming.

In [23]:
query.stop()

##  Download del file

Scarichiamo il file CSV generato.

In [24]:
from google.colab import files
files.download("sensors.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusione

In questo progetto abbiamo utilizzato Apache Spark per generare e analizzare dati IoT. Abbiamo creato un dataset simulato, eseguito analisi con Spark SQL e simulato un flusso di dati in tempo reale tramite Spark Structured Streaming.